## Notebook Stack

### Input Output
`pyarrow` is the defacto parquet and arrow standard library tooling in the python eco-system, offering exceptional performance, documentation and stability.

Its `fs` class allows connecting to parquet datasets in AWS S3, which is perfect for our use case.

### Computation
`polars` is fast replacing pandas as the primary python dataframe API. It is more tightly integrated with arrow (and pyarrow), has exceptional performance, greater than memory dataset manipulation, a more consistent and declarative API, built in plotting and many other advantages.

### Mapping
`pydeck` is a python binding for `deck.gl`, a GPU-powered framework for visual exploratory data analysis of large datasets. It is optimised for Jupyter notebooks and highly customisable.

Standard mapping tools like `folium` struggle to handle the number of mapped shapes, so we favor the more optimised `pydeck` library.

## Connecting to Cloud Optimised Datasets

NESP 5.9 datasets can be connected to via a pyarrow `S3FileSystem`.

The `S3FileSystem` manages the connection to the NESP 5.9 datasets stored in S3, resulting in a `pyarrow.Dataset` handle that we may use for efficient loading and subsetting of the dataset.

Note that a dataset can be comprised of a single parquet file or partitioned (into separate parquet fragments/files).

This step also makes the dataset schema available and file structure available allowing for pre-filtering without downloading the entire dataset.

In [1]:
import pyarrow
import pyarrow.fs
import pyarrow.dataset
import pyarrow.parquet

# Construct the anonymous file system responsible for reading from the public S3 bucket
FILE_SYSTEM = pyarrow.fs.S3FileSystem(
    region="ap-southeast-2", 
    anonymous=True,
)

DATASET_PATH = "aodn-cloud-optimised/diver_photoquadrat_qc_DwC.parquet"
SCHEMA_PATH = f"{DATASET_PATH}/_common_metadata"

# Create the dataset connection
# By convention, datasets are labelled `ds`
ds = pyarrow.dataset.dataset(
    source=DATASET_PATH,
    filesystem=FILE_SYSTEM,
    schema=pyarrow.parquet.read_schema(
        where=SCHEMA_PATH,
        filesystem=FILE_SYSTEM,
    ),
    partitioning="hive",
)

# The comprising files of the dataset can be inspected
print(f"There are `{len(list(ds.get_fragments()))}` cloud optimised file fragments.")

There are `19` cloud optimised file fragments.


In [2]:
ds.schema

class: string
  -- field metadata --
  url: 'http://rs.tdwg.org/dwc/terms/class'
  type: 'string'
  nullable: 'True'
  long_name: 'Class'
  definition: 'The full scientific name of the class in which the dwc:Tax' + 17
genus: string
  -- field metadata --
  url: 'http://rs.tdwg.org/dwc/terms/genus'
  type: 'string'
  nullable: 'True'
  long_name: 'Genus'
  definition: 'The full scientific name of the genus in which the dwc:Tax' + 17
order: string
  -- field metadata --
  url: 'http://rs.tdwg.org/dwc/terms/order'
  type: 'string'
  nullable: 'True'
  long_name: 'Order'
  definition: 'The full scientific name of the order in which the dwc:Tax' + 17
family: string
  -- field metadata --
  url: 'http://rs.tdwg.org/dwc/terms/family'
  type: 'string'
  nullable: 'True'
  long_name: 'Family'
  definition: 'The full scientific name of the family in which the dwc:Ta' + 18
phylum: string
  -- field metadata --
  url: 'http://rs.tdwg.org/dwc/terms/phylum'
  type: 'string'
  nullable: 'True'
  long

In [3]:
import polars
import polars_h3

df = polars.DataFrame(
    data=ds.to_table(),
)
display(df)
display(df["eventID"].value_counts())

class,genus,order,family,phylum,eventID,h3Index,kingdom,eventDate,taxonRank,RLSLineage,occurrenceID,basisOfRecord,catamiLineage,scientificName,associatedMedia,decimalLatitude,individualCount,decimalLongitude,occurrenceStatus,scientificNameID,RLSIdentification,catamiIdentification,scientificNameAuthorship,australianMarineRegionsTags,polygon,filename,timestamp
str,str,str,str,str,str,str,str,date,str,str,str,str,str,str,str,f64,i32,f64,str,str,str,str,str,str,str,str,i64
null,null,null,null,null,"""7000597_8_2226""","""84bf4c1ffffffff""",null,2008-01-09,null,"""Physical > Substrate > Bare Ro…","""1330""","""MachineObservation""","""2 Physical > Substrate > Conso…",null,"""https://squidle.org/iframe/api…",-42.59,1,148.05,"""present""",null,"""Bare Rock""","""Rock""",null,"""IMCRA:Provincial:Name:Tasmania…","""010300000001000000050000000000…","""nrmn.parquet""",1199145600
"""Ulvophyceae""","""Caulerpa""","""Bryopsidales""","""Caulerpaceae""","""Chlorophyta""","""7000597_8_2226""","""84bf4c1ffffffff""","""Plantae""",2008-01-09,"""Genus""","""Biota > Macroalgae > Medium fo…","""1331""","""MachineObservation""","""1.1 Biota > Macroalgae > Erect…","""Caulerpa""","""https://squidle.org/iframe/api…",-42.59,1,148.05,"""present""","""urn:lsid:marinespecies.org:tax…","""Caulerpa""","""Green""","""J.V.Lamouroux, 1809""","""IMCRA:Provincial:Name:Tasmania…","""010300000001000000050000000000…","""nrmn.parquet""",1199145600
"""Florideophyceae""",null,"""Corallinales""",null,"""Rhodophyta""","""7000597_8_2226""","""84bf4c1ffffffff""","""Plantae""",2008-01-09,"""Order""","""Biota > Macroalgae > Crustose …","""1332""","""MachineObservation""","""1.1 Biota > Macroalgae > Encru…","""Corallinales""","""https://squidle.org/iframe/api…",-42.59,9,148.05,"""present""","""urn:lsid:marinespecies.org:tax…","""Crustose coralline algae""","""Calcareous""","""P.C. Silva & H.W. Johansen, 19…","""IMCRA:Provincial:Name:Tasmania…","""010300000001000000050000000000…","""nrmn.parquet""",1199145600
"""Phaeophyceae""","""Ecklonia""","""Laminariales""","""Lessoniaceae""","""Ochrophyta""","""7000597_8_2226""","""84bf4c1ffffffff""","""Chromista""",2008-01-09,"""Species""","""Biota > Macroalgae > Macroalga…","""1333""","""MachineObservation""","""1.1 Biota > Macroalgae > Large…","""Ecklonia radiata""","""https://squidle.org/iframe/api…",-42.59,7,148.05,"""present""","""urn:lsid:marinespecies.org:tax…","""Ecklonia radiata""","""Large canopy-forming""","""(C.Agardh) J.Agardh, 1848""","""IMCRA:Provincial:Name:Tasmania…","""010300000001000000050000000000…","""nrmn.parquet""",1199145600
null,null,null,null,null,"""7000597_8_2226""","""84bf4c1ffffffff""",null,2008-01-09,"""Superdomain""","""Biota > Macroalgae > Encrustin…","""1334""","""MachineObservation""","""1.1 Biota""","""Biota""","""https://squidle.org/iframe/api…",-42.59,2,148.05,"""present""","""urn:lsid:marinespecies.org:tax…","""Encrusting leathery algae""","""1.1 Biota""",null,"""IMCRA:Provincial:Name:Tasmania…","""010300000001000000050000000000…","""nrmn.parquet""",1199145600
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
null,null,null,null,null,"""923406227_91_19315""","""84bf4adffffffff""",null,2026-03-12,"""Superdomain""","""Biota > Macroalgae > Encrustin…","""120469""","""MachineObservation""","""1.1 Biota""","""Biota""","""https://squidle.org/iframe/api…",-40.89,1,148.26,"""present""","""urn:lsid:marinespecies.org:tax…","""Encrusting leathery algae""","""1.1 Biota""",null,"""IMCRA:Mesoscale:Water Type:Col…","""010300000001000000050000000000…","""nrmn.parquet""",1767225600
"""Phaeophyceae""","""Macrocystis""","""Laminariales""","""Laminariaceae""","""Ochrophyta""","""923406227_91_19315""","""84bf4adffffffff""","""Chromista""",2026-03-12,"""Species""","""Biota > Macroalgae > Macroalga…","""120470""","""MachineObservation""","""1.1 Biota > Macroalgae > Large…","""Macrocystis pyrifera""","""https://squidle.org/iframe/api…",-40.89,11,148.26,"""present""","""urn:lsid:marinespecies.org:tax…","""Macrocystis""

eventID,count
str,u32
"""4000733_11_739""",12
"""912354687_8_7900""",10
"""912343342_6_0""",11
"""912340188_8_3905""",10
"""912350270_11_1696""",15
…,…
"""912352225_8_4015""",7
"""912342284_11_975""",5
"""912343826_6_0""",12


In [4]:
import pydeck
import util

# Aggregate and generate h3 layers
h3_layers = util.generate_pydeck_hexagon_layers(
    df=(

        # H3 Aggregated dataframe with dataset and number of records attributes
        df
        .group_by(
            polars.col("h3Index"),
        )
        .agg(
            polars.col("eventID").count().alias("n_records"),
        )
    ),
    aggregate_column_name="n_records",
    n_quantiles = 10,
    color_palette="plasma",
)

# Run the map
deck = pydeck.Deck(
    layers=h3_layers,
    map_style=pydeck.map_styles.LIGHT_NO_LABELS,
    tooltip=util.TOOLTIP,
    initial_view_state=pydeck.ViewState(
        latitude=-30,
        longitude=135,
        zoom=2.5,
    ),
)
deck.show()

# Finding Trend Sites with H3
We may identify trend sites by finding a H3 cell associated with data over a time range.

For example, cell `84be639ffffffff` (situated over Port Phillip Bay) has over 5000 records.

We may aggregate the `individualCount` per `scientificName` and normalize by yearly total observations to generate a time series analysis.

In [5]:
import altair as alt

# Filter to Port Phillip Bay
_84be639ffffffff_df = (
    df
    .filter(
        polars.col("h3Index").eq("84be639ffffffff"),
    )
)

# Generate yearly aggregates
scientific_name_aggregated_84be639ffffffff_df = (
    _84be639ffffffff_df.group_by("scientificName", polars.col("eventDate").dt.year().alias("year"))
    .agg(
        polars.col("individualCount").sum().alias("species_count")
    )
    .with_columns(
        (polars.col("species_count") / polars.col("species_count").sum().over("year")).alias("year_proportion")
    )
    .sort("year_proportion", descending=True)
    .sort("year")
)

# Plot the year proportion of each `scientificName`
faceted_chart = (
    alt.Chart(scientific_name_aggregated_84be639ffffffff_df)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("year_proportion:Q", title="Share", axis=alt.Axis(format="%")),
        color=alt.Color("scientificName:N", legend=None),
        tooltip=[
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("species_count:Q", title="Count"),
            alt.Tooltip("year_proportion:Q", title="Share", format=".2%"),
        ],
    )
    .properties(width=220, height=150)
    .facet(
        facet=alt.Facet("scientificName:N", title="Scientific Name"),
        columns=3
    )
    .resolve_scale(y="independent")  # Change to 'shared' if you want a uniform Y-axis scale
)

faceted_chart.show()

alt.FacetChart(...)